# Harmonia v0.2 — Bayesian dose-response uncertainty quantification

Where v0.1 made input variability a first-class **field**, v0.2 makes it a
first-class **inference**: the spread of an IC50 stops being a number transcribed
from a table and becomes a posterior, derived under a declared prior from the
underlying data, with the unidentifiable case (max block < ~60%) handled as a
**one-sided censored posterior** rather than a binary exclusion.

> Harmonia is **NOT** a clinical tool and **NOT** a regulatory safety
> determination. A posterior is never a permission to point-estimate; the outputs
> are distributions, flip frequencies, and explicitly-labeled diagnostics — never a
> verdict (spec.md §10; spec v0.2 §10).

Executed in CI under `nbmake`.

In [ ]:
import numpy as np
import harmonia
from harmonia.infer import posterior, resolve_prior, learn_tau_pop, infer_channel
from harmonia.records import ChannelBlock

ds = harmonia.load()
print(harmonia.CLINICAL_USE)
prior = resolve_prior(ds)
tau_pop = learn_tau_pop(ds.channel_blocks, prior)
print(f"prior={prior.id}  learned between-lab SD tau_pop={tau_pop:.3f} log10")

## 1. The prior is a declared input, not a hidden choice

Every posterior names the version-pinned prior it was inferred under, and reports
its `prior_sensitivity` so a prior-dominated number is visible (spec v0.2 §7).

In [ ]:
d = prior.raw
print(f"channel-level true log10 IC50 ~ Normal(m0={prior.m0}, s0={prior.s0})")
print(f"between-lab SD ~ HalfNormal(scale={prior.tau_scale})   hill ~ LogNormal({prior.hill_mu}, {prior.hill_sigma})")
print(f"predictive={d['predictive']}   citations={d['citations']}")

## 2. A well-identified channel: the posterior reduces to the v0.1 answer

Non-drift guarantee (spec v0.2 §0.2): in the multi-source, agreed regime the
posterior mean of `log10(IC50)` converges to the log-geomean the v0.1 sampler
centers on. dofetilide hERG has three agreeing labs.

In [ ]:
b = next(x for x in ds.blocks_for('dofetilide') if x.channel == 'IKr')
log_geomean = float(np.log10(np.exp(np.mean(np.log(b.source_ic50s_nm)))))
p = posterior(ds, 'dofetilide', 'IKr', n_draws=4000, seed=0)
print(f"sources (nM): {b.source_ic50s_nm}")
print(f"log-geomean (v0.1 center): {log_geomean:.3f}")
print(f"posterior mean log10 IC50: {p.mean_log10:.3f}  (sd {p.sd_log10:.3f})")
print(f"hill posterior: {p.hill_mean:.2f} +/- {p.hill_sd:.2f}   (v0.1 fixed it)")
print(f"identifiability={p.identifiability_score:.2f}  prior_sensitivity={p.prior_sensitivity:.2f}  rhat={p.rhat_max:.4f}")
assert abs(p.mean_log10 - log_geomean) < 0.05

## 3. A single-source channel borrows the dataset-learned spread

v0.1 gave a single-source channel a hard-coded `DEFAULT_SINGLE_SOURCE_SIGMA`.
v0.2 replaces that magic constant with `tau_pop` — the pooled between-lab spread
*learned across every multi-source channel in the dataset* (spec v0.2 §2.2).

In [ ]:
single = next(x for x in ds.channel_blocks
              if isinstance(x, ChannelBlock) and x.identifiable and len(x.source_ic50s_nm) == 1)
ps = infer_channel(single, prior, tau_pop, n_draws=4000, seed=0)
print(f"{single.id}  (single source)")
print(f"posterior sd_log10={ps.sd_log10:.3f}  vs learned tau_pop={tau_pop:.3f}")
print("the spread is an inferred, citable quantity, not a constant")

## 4. The censored (unidentifiable) channel: information bounded, not discarded

A sub-60%-block measurement does not identify the IC50, but it *bounds* it from
below near the top tested dose. v0.2 carries that as a one-sided posterior with a
heavy, prior-shaped right tail — prior-dominated by construction, and still
Tier-D-capped (spec v0.2 §2.3, §6).

In [ ]:
cens = next(x for x in ds.channel_blocks if isinstance(x, ChannelBlock) and not x.identifiable)
pc = infer_channel(cens, prior, tau_pop, n_draws=6000, seed=0)
q = 10 ** np.quantile(pc.log10_ic50, [0.05, 0.5, 0.95])
print(f"{cens.id}  max block {cens.assay_context.max_block_observed_percent}%  (recovered top dose ~ {pc.x_max_nm:.0f} nM)")
print(f"IC50 posterior q05/median/q95 = {q[0]:.0f} / {q[1]:.0f} / {q[2]:.0f} nM  (heavy right tail)")
print(f"identifiability={pc.identifiability_score:.2f}  prior_sensitivity={pc.prior_sensitivity:.2f}  prior_dominated={pc.prior_dominated}")
assert pc.censored and pc.prior_dominated

## 5. Two distinct flip frequencies

The headline flip frequency samples the **true-value** posterior (μ_c). The new
`reproducibility_flip_frequency` samples the **new-lab predictive** (adds the
between-lab spread τ_c) — "how much would a fresh replication move the call?"
Both are distributions; neither is a verdict (spec v0.2 §2.4).

In [ ]:
for drug in ['dofetilide', 'verapamil']:
    a = harmonia.assess(ds, drug, n_mc=80, uq='bayes', seed=0)
    mix = {k: round(v, 2) for k, v in a.classification_distribution.items() if v}
    print(f"{drug:<12} point={a.classification.upper():<13} flip(true)={a.classification_flip_frequency:.0%}  "
          f"flip(new-lab)={a.reproducibility_flip_frequency:.0%}  {mix}")

## 6. Global, interaction-aware sensitivity (Sobol)

The v0.1 one-at-a-time attribution reads main effects but cannot see channel
interactions. The variance-based (Sobol) design reports first-order `S_i`,
total-effect `S_Ti`, and the **interaction load** `S_Ti - S_i` with Monte-Carlo
standard errors (spec v0.2 §5).

In [ ]:
s = harmonia.flip_sensitivity(ds, 'verapamil', method='sobol', n_mc=64, seed=0)
print(f"dominant (interaction-aware total-effect) driver: {s.dominant_channel}")
for c in s.channels:
    print(f"  {c.channel:<6} S_i={c.first_order:.2f}+/-{c.first_order_se:.2f}  "
          f"S_Ti={c.total_effect:.2f}+/-{c.total_effect_se:.2f}  interaction={c.interaction_load:+.2f}")

*v0.2 changes how Harmonia computes its uncertainty, not what it is willing to
claim. The output is still a distribution and a flip frequency, the 60%-block gate
is still hard, and the answer is still never a verdict — only now the spread it
propagates is inferred from the data under a declared prior, and the information in
an unidentifiable measurement is bounded and carried rather than discarded.*